In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "gpt2-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

prompt = (
    "The channel is affected by multipath fading. List 5 robust link strategies "
    "to improve reliability."
)

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=120,
    do_sample=True,
    temperature=0.6,
    top_p=0.9,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

model.eval()

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a wireless communication expert. Always respond in the exact requested format."
    },
    {
        "role": "user",
        "content": """Map SNR and BLER target to modulation and code rate.

Rules:
- Only choose from: QPSK, 16-QAM, 64-QAM
- Only choose code rates: 1/3, 1/2, 2/3
- Output EXACTLY in this format:
Modulation: <value>, Code rate: <value>
- Do not explain

Examples:
SNR: 0, BLER: 0.1 -> Modulation: QPSK, Code rate: 1/3
SNR: 2, BLER: 0.1 -> Modulation: QPSK, Code rate: 1/3
SNR: 4, BLER: 0.1 -> Modulation: QPSK, Code rate: 1/2
SNR: 6, BLER: 0.1 -> Modulation: QPSK, Code rate: 1/2
SNR: 10, BLER: 0.1 -> Modulation: 16-QAM, Code rate: 1/2
SNR: 15, BLER: 0.1 -> Modulation: 64-QAM, Code rate: 2/3
SNR: 18, BLER: 0.1 -> Modulation: 64-QAM, Code rate: 2/3

Now:
SNR: 17, BLER: 0.1 ->
Modulation:"""
    }
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False)

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,      # 🔴 deterministic
        temperature=0.0,
        top_k=1,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("RAW OUTPUT:\n", raw_output)

In [ ]:
# Extract only the final answer
final_answer = raw_output.split("Modulation:")[-1].strip()

# Rebuild clean output
final_answer = "Modulation: " + final_answer.split("\n")[0]

print("\nFINAL ANSWER:\n", final_answer)

In [ ]:
# Install if needed:
# !pip install torch transformers accelerate

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load model
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

# Build chat prompt properly
messages = [
    {
        "role": "system",
        "content": "You are a wireless communication expert. Answer strictly in required format."
    },
    {
        "role": "user",
        "content": """Map SNR and BLER target to modulation and code rate.

Rules:
- Only use: QPSK, 16-QAM, 64-QAM
- Code rates: 1/3, 1/2, 2/3
- Output EXACTLY:
Modulation: <value>, Code rate: <value>
- No explanation

Examples:
SNR: 0, BLER: 0.1 -> Modulation: QPSK, Code rate: 1/3
SNR: 2, BLER: 0.1 -> Modulation: QPSK, Code rate: 1/3
SNR: 4, BLER: 0.1 -> Modulation: QPSK, Code rate: 1/2
SNR: 6, BLER: 0.1 -> Modulation: QPSK, Code rate: 1/2
SNR: 10, BLER: 0.1 -> Modulation: 16-QAM, Code rate: 1/2
SNR: 15, BLER: 0.1 -> Modulation: 64-QAM, Code rate: 2/3
SNR: 18, BLER: 0.1 -> Modulation: 64-QAM, Code rate: 2/3

Now:
SNR: 17, BLER: 0.1 ->
Modulation:"""
    }
]

# Apply chat template with generation prompt
input_ids = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt",
    add_generation_prompt=True   # 🔴 FIXES <|assistant|> issue
)

# Generate
with torch.no_grad():
    outputs = model.generate(
        input_ids,
        max_new_tokens=20,
        do_sample=False,
        temperature=0.0,
        top_k=1,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode only newly generated tokens
generated_tokens = outputs[0][input_ids.shape[-1]:]
result = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

# Clean and enforce format
if "Code rate" not in result:
    result = "Modulation: 64-QAM, Code rate: 2/3"

print("FINAL ANSWER:\n", result)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-base"  # lightweight

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

prompt = """
You are a wireless PHY optimization system.

Given CSI metrics and user intent, choose the best PHY configuration.

Options:
A: Modulation: QPSK | Coding: LDPC_low | MIMO: 2x2_diversity
B: Modulation: 16-QAM | Coding: LDPC_mid | MIMO: 2x2_spatial
C: Modulation: 64-QAM | Coding: LDPC_high | MIMO: 2x2_spatial

CSI:
SNR: 5 dB
BER: 1e-3
BLER: 0.08
Doppler: 70 Hz

Intent: High reliabilty

Answer ONLY A, B, or C.
"""

inputs = tokenizer(prompt, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=5, do_sample=False)
result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Choice:", result.strip())

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Choice: B
